# J0.5 — la chaîne complète en six lignes

**IAA1302.1 — Apprentissage Automatique · ESGC-VAK 2026-2027**

> **Avant ce notebook :** `notebook_J0.ipynb` — les fondamentaux Python.
> Si vous n'avez jamais écrit une ligne de Python, commencez par lui.

Ce notebook fait **tout le trajet** : lire les données, séparer la cible, entraîner, prédire,
écrire un fichier de soumission, le déposer. Aucune explication de ce qui se passe à
l'intérieur : on regarde **ce qui entre** et **ce qui sort**.

## Les trois règles du dépôt

1. **`Save & Run All`** (version sauvegardée, exécutée de bout en bout, état propre).
2. **Privé** : le notebook n'est visible que de vous et de l'enseignant.
3. **CPU, moins de 10 minutes.** Le GPU est un bonus, jamais un prérequis.

## À déposer à la fin de la séance

- le **lien** de ce notebook (privé, exécuté) ;
- votre **tableau de 3 scores** : baseline naïve, score interne, score public.

---

In [1]:
# 0 — les bibliothèques du cours, et les versions pour lesquelles il a été écrit :
#        pandas 3.0.6 · numpy 2.4.3 · scikit-learn 1.9.0 · jinja2
#
# Le code tourne AUSSI sous pandas 2 (c'est ce que Kaggle fournit) : les deux sont testés.
#
# SUR KAGGLE : rien à faire. Les bibliothèques sont déjà installées et l'INTERNET EST COUPÉ :
# une commande d'installation y échoue. Ce n'est pas une erreur de votre part.
#
# EN LOCAL, une seule fois — dans un terminal :
#     pip install "pandas==3.0.6" "numpy==2.4.3" "scikit-learn==1.9.0" "jinja2"
# dans une cellule d'un notebook local, la même chose avec un point d'exclamation devant.

# Ce que vous avez, à cet instant :
import importlib

for nom in ("pandas", "numpy", "sklearn", "jinja2"):
    try:
        print(f"{nom:<8}", importlib.import_module(nom).__version__)
    except ImportError:
        print(f"{nom:<8} absent — voir les lignes d'installation ci-dessus")

pandas   2.3.3
numpy    2.0.2
sklearn  1.6.1
jinja2   3.1.6


In [2]:
# 1 — les outils
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier

In [3]:
# 2 — où sont les fichiers (le notebook marche sur Kaggle ET en local)
# On CHERCHE train.csv au lieu de supposer un chemin. Kaggle a déjà changé ses points de
# montage (/kaggle/input/<identifiant> puis /kaggle/input/competitions/<identifiant>) : un
# chemin recopié à la main se périme, une recherche non.
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie", os.path.join("..", "sortie"),
                            os.path.join("..", "Donnees_competition", "sortie")])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"

CHEMIN_TRAIN = os.path.join(DOSSIER, "train.csv")
CHEMIN_TEST = os.path.join(DOSSIER, "test.csv")

print("le dossier des données trouvé :", DOSSIER)
print("  fichier d'entraînement :", CHEMIN_TRAIN)
print("  fichier de test        :", CHEMIN_TEST)

le dossier des données trouvé : /kaggle/input/competitions/rappels-et-sur-réservation-des-créneaux
  fichier d'entraînement : /kaggle/input/competitions/rappels-et-sur-réservation-des-créneaux/train.csv
  fichier de test        : /kaggle/input/competitions/rappels-et-sur-réservation-des-créneaux/test.csv


In [4]:
# 3 — lire, séparer la cible, garder de côté
train = pd.read_csv(CHEMIN_TRAIN, low_memory=False)
test = pd.read_csv(CHEMIN_TEST, low_memory=False)

ID = "ligne_id"
CIBLE = "patient_absent"

# Ce que ce notebook met de côté — volontairement, et sans l'expliquer aujourd'hui :
#   · les identifiants et les colonnes connues APRÈS l'issue (fuite) ;
#   · les dates brutes : une date n'est pas un nombre — S10 montrera comment la transformer ;
#   · `taille_cm` et `date_naissance`, saisies en TEXTE — S2 montrera comment les reprendre.
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance"]

X = train.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in train.columns])
y = train[CIBLE]

# Les colonnes non numériques sont déclarées catégorielles. Détection par `is_numeric_dtype`
# et non par `dtype == "object"` : pandas 3 a un dtype `str` dédié, pandas 2 un `object`.
CAT = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]

# Une colonne à très forte cardinalité n'est pas une catégorielle : les modèles d'arbres
# acceptent les catégorielles natives jusqu'à 255 modalités. Au-delà, il faut un autre
# traitement (regroupement, encodage) — S10. On l'écarte ici, et on dit pourquoi.
TROP_FIN = [c for c in CAT if X[c].nunique() > 255]
if TROP_FIN:
    print("écartée(s), cardinalité > 255 :", [(c, int(X[c].nunique())) for c in TROP_FIN])
X = X.drop(columns=TROP_FIN)
CAT = [c for c in CAT if c not in TROP_FIN]

for c in CAT:
    X[c] = X[c].fillna("").astype("category")

print("après lecture \n")
print("cible :", round(100 * y.mean(), 2), "% de positifs")
print("colonnes catégorielles :", CAT)

écartée(s), cardinalité > 255 : [('tournee_jour', 9000)]
après lecture 

cible : 18.19 % de positifs
colonnes catégorielles : ['clinique', 'district', 'sexe', 'quartier', 'assurance', 'rappel_envoye', 'zone_clinique', 'canal_prise']


## 4. Regardons les deux tableaux

`train` et `test` ont la même forme générale. Le plus simple est de les **afficher** :

In [5]:
# 4 — les deux tableaux, et leurs dimensions
print("les dimensions, en lignes et colonnes :")
print("train :", train.shape)
print("test  :", test.shape)

print("les cinq premières lignes de train \n")
train.head()

les dimensions, en lignes et colonnes :
train : (149894, 39)
test  : (63721, 38)
les cinq premières lignes de train 



,dossier_id,ligne_id,patient_id,clinique,district,age,sexe,quartier,assurance,hypertension,...,indice_risque_externe,indice_meteo_jour,zone_clinique,tournee_jour,cout_acte,age_mois,score_territorial,canal_prise,taux_absence_passe,patient_absent
0,DOS-0023742,23743,P070425,C02,D3,53,F,Q04,privée,0,...,46.47,56,urbaine,T00000,13.1,636,-1.1621,guichet,0.0,0
1,DOS-0012153,12154,P109083,C36,D6,38,F,Q01,mutuelle,0,...,39.64,46,urbaine,T00002,6.6,456,-0.0963,guichet,0.0,0
2,DOS-0045022,45023,P036240,C43,D5,35,M,Q52,publique,1,...,48.87,44,urbaine,T00004,16.2,420,0.7951,téléphone,0.0,0
3,DOS-0143858,143859,P033484,C10,D2,36,M,Q07,aucune,0,...,41.66,59,urbaine,T00005,21.1,432,0.3788,guichet,0.0,0
4,DOS-0183885,183886,P035934,C28,D6,32,M,Q02,aucune,0,...,69.87,31,urbaine,T00006,27.0,384,1.2562,guichet,0.0,0


Le second ressemble au premier, à une colonne près. Affichez-le de la même façon :

In [6]:
# 5 — le second tableau
print("les cinq premières lignes de test \n")
test.head()

les cinq premières lignes de test 



,dossier_id,ligne_id,patient_id,clinique,district,age,sexe,quartier,assurance,hypertension,...,montant_acte,indice_risque_externe,indice_meteo_jour,zone_clinique,tournee_jour,cout_acte,age_mois,score_territorial,canal_prise,taux_absence_passe
0,DOS-0127993,127994,P080352,C30,D6,32,F,Q02,publique,0,...,197.03,47.75,69,urbaine,T00001,14.4,384,0.5662,NaN,0.00
1,DOS-0188717,188718,P086289,C21,D4,65,F,Q50,publique,0,...,378.78,38.95,29,urbaine,T00003,27.7,780,0.2251,NaN,0.00
2,DOS-0033101,33102,P061159,C77,D2,54,F,Q01,aucune,0,...,132.47,45.70,60,urbaine,T00008,9.7,648,-0.5043,NaN,1.00
3,DOS-0102131,102132,P000440,C18,D5,47,F,Q03,publique,0,...,284.38,66.10,62,urbaine,T00010,20.8,564,-0.7630,NaN,0.25
4,DOS-0081357,81358,P118098,C01,D1,70,F,Q03,mutuelle,0,...,258.17,43.96,46,urbaine,T00018,18.9,840,0.2524,NaN,0.00


#### De quoi s'agit-il, exactement

- **`train`** — une ligne par rendez-vous, **avec** la réponse : on sait s'il a été manqué.
- **`test`** — une ligne par rendez-vous, **sans** la réponse : c'est ce qu'on vous demande de deviner.

Et voilà la différence de colonnes, mesurée :

In [7]:
# 6 — les colonnes, côte à côte
colonnes_train = set(train.columns)
colonnes_test = set(test.columns)

print("combien de colonnes :")
print("dans train :", len(colonnes_train))
print("dans test  :", len(colonnes_test))

print("la différence \n")
print("dans train, absente de test :", sorted(colonnes_train - colonnes_test))
print("dans test, absente de train :", sorted(colonnes_test - colonnes_train))

combien de colonnes :
dans train : 39
dans test  : 38
la différence 

dans train, absente de test : ['patient_absent']
dans test, absente de train : []


### La colonne qui manque

`test` a **une colonne de moins**, et c'est toujours la même : `patient_absent`.

Ce n'est pas un défaut du fichier : **c'est l'exercice**. `train` porte la réponse, `test` ne la porte pas — sinon il n'y aurait rien à prédire. Vérifiez-le en une ligne :

In [8]:
# 7 — la colonne manquante est bien celle qu'on prédit
print("la cible est dans train :", CIBLE in train.columns)
print("la cible est dans test  :", CIBLE in test.columns)

la cible est dans train : True
la cible est dans test  : False


*Deux questions restent ouvertes, et elles resteront : pourquoi `test` a-t-il moins de lignes que `train`, et comment cette répartition a-t-elle été faite ? On y reviendra.*

In [9]:
# 8 — la baseline naïve : « personne ne manque son rendez-vous »
baseline = 100 * max(y.mean(), 1 - y.mean())

print("la référence : ce que vaut un modèle qui répond toujours la même chose \n")
print("baseline majoritaire :", round(baseline, 2), "% d'accuracy")

la référence : ce que vaut un modèle qui répond toujours la même chose 

baseline majoritaire : 81.81 % d'accuracy


In [10]:
# 9 — entraîner sur une partie, mesurer sur l'autre (le score INTERNE)
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0)
modele.fit(Xtr, ytr)

print("le score interne, mesuré sur des données que le modèle n'a pas vues \n")
print("score interne :", round(100 * modele.score(Xva, yva), 2), "%")

le score interne, mesuré sur des données que le modèle n'a pas vues 

score interne : 82.81 %


In [11]:
# 10 — prédire sur le test et écrire le fichier de soumission
Xt = test.drop(columns=[c for c in A_ECARTER + TROP_FIN if c in test.columns])

# MÊMES catégories que l'entraînement, sinon décalage silencieux
for c in CAT:
    Xt[c] = Xt[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)

soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)

print("le fichier écrit : submission.csv \n")
print("dimensions :", soumission.shape, "(lignes, colonnes)")
print("les cinq premières lignes \n")
soumission.head()

le fichier écrit : submission.csv 

dimensions : (63721, 2) (lignes, colonnes)
les cinq premières lignes 



,ligne_id,patient_absent
0,127994,0
1,188718,0
2,33102,0
3,102132,0
4,81358,0


---

## Et maintenant

1. Cliquez sur **Submit** (à droite) et déposez `submission.csv`.
2. Notez les **trois scores** : baseline naïve, score interne, score public.
3. Regardez l'écart entre votre score interne et votre score public. Il est **moche**.
   C'est normal — et c'est le sujet de la séance 3.

> **Ce que ce notebook ne fait pas :** il ne vous dit pas *pourquoi* ça marche. Il ne vous
> explique ni le gradient, ni la fonction de coût, ni ce que « boosting » veut dire.
> Ces questions ont un nom, et une séance y répondra — mais pas aujourd'hui.

**Livrable à déposer :** le lien de ce notebook exécuté + votre tableau de 3 scores.

### Ce que vous venez de faire — et comment ça s'appelle

- `patient_absent` est **la cible** : ce que le modèle doit deviner.
- Le score de la classe majoritaire est **une référence**. Sans elle, aucun score ne veut rien
  dire — c'est ce que vous comparerez toute l'année.
- Les colonnes mises de côté « parce qu'on ne les a pas encore vues » sont des **fuites** : elles
  n'existeraient pas au moment où la décision se prend. On y revient en S2.
- `tournee_jour` a été écartée : trop de modalités pour un modèle d'arbres, qui plafonne à 255.
  C'est **la cardinalité**. Développé en S10.

**Rien de tout ça ne vous est expliqué aujourd'hui.** Vous avez vu ce qui entre et ce qui sort ;
les noms viendront quand les séances les rendront nécessaires.